# Prior Filtering
- "Modified Rankin Scale at 90 days": selection of instances with completed records (non-missing values)
- "Diagnostic impression": selection of acute ischemic stroke (AIS)

In [ ]:
# ==============================================================================
# 0. ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
!pip install optuna catboost lightgbm xgboost fairlearn boruta openpyxl -q

import os
import sys
import re
import warnings
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import optuna

# Compatibility patches
import sklearn.utils._array_api
if not hasattr(sklearn.utils._array_api, '_logsumexp'):
    from scipy.special import logsumexp
    sklearn.utils._array_api._logsumexp = logsumexp
if not hasattr(sklearn.utils._array_api, '_half_multinomial_loss'):
    sklearn.utils._array_api._half_multinomial_loss = lambda *args, **kwargs: 0.0

import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                             precision_score, recall_score, confusion_matrix)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from tqdm.auto import tqdm

sklearn.set_config(transform_output="pandas")
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
plt.style.use('default')

class HiddenPrints:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

# ==============================================================================
# 1. CONFIGURATION AND DATA INGESTION
# ==============================================================================
START_SEED = 100
END_SEED = 101

FILE_PATH = '/content/drive/MyDrive/avc/seed/avci.xlsx'
OUTPUT_DIR = '/content/drive/MyDrive/avc/seed/outputs/clinical_stress_test'
MASTER_CSV_PATH = f"{OUTPUT_DIR}/Comprehensive_MultiSeed_Master_Log.csv"
FEATURES_LOG_PATH = f"{OUTPUT_DIR}/Feature_Selection_Log.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots_models", exist_ok=True)

print("="*90)
print(f"INITIALIZING MULTI-ALGORITHM STRESS TEST (BATCH: SEEDS {START_SEED} TO {END_SEED-1})")
print("="*90)

df = pd.read_excel(FILE_PATH)
df.columns = [re.sub(r'[^a-z0-9_]+', '_', str(c).lower()).strip('_') for c in df.columns]

# Target Formatting
df = df.dropna(subset=['escala_de_rankin_modificada_em_90_dias'])
df['target_binary'] = np.where(df['escala_de_rankin_modificada_em_90_dias'] <= 3, 0, 1)

# Leakage Prevention
cols_to_drop = [
    'record_id', 'nome_do_profissional', 'raca_cor_autodeclarada',
    'escala_de_rankin_modificada_em_90_dias', 'novo_nihss', 'rankin_1',
    'laudo_da_tomografia_computadorizada_de_controle_24_horas_ap_s_tromb_lise',
    'necessidade_de_re_acionamento_36h_por_suspeita_de_transforma_o_hemorr_gica_ap_s_trombolise',
    'o_paciente_ou_respons_vel_legal_consentiu_no_compartilhamento_de_seus_dados_pessoais_e_telefone_para_contato_em_90_dias',
    'data_e_hora', 'data_de_nascimento', 'ltima_ocupa_o_pr_via_ao_avc', 'unidade_1', 'cod_profissional',
    'realizada_tromb_lise_endovenosa', 'data_e_hor_rio_do_bolus_do_trombol_tico',
    'condutas_sugeridas_foram_realizadas', 'tromb_lise_endovenosa', 'trombectomia_mec_nica', 'impress_o_diagn_stica'
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors='ignore')

# Date Features
date_cols = [c for c in df.columns if 'data' in c or 'hor_rio' in c]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

if 'data_e_hor_rio_da_chegada_no_hospital_sat_lite' in df.columns:
    df['arrival_hour'] = df['data_e_hor_rio_da_chegada_no_hospital_sat_lite'].dt.hour
    df['arrival_day_of_week'] = df['data_e_hor_rio_da_chegada_no_hospital_sat_lite'].dt.dayofweek

if 'data_e_hor_rio_da_chegada_no_hospital_sat_lite' in df.columns and 'data_e_hor_rio_do_in_cio_dos_sintomas' in df.columns:
    df['delta_symptom_to_door_min'] = (df['data_e_hor_rio_da_chegada_no_hospital_sat_lite'] - df['data_e_hor_rio_do_in_cio_dos_sintomas']).dt.total_seconds() / 60

delta_cols = [c for c in df.columns if 'delta' in c]
for c in delta_cols:
    df[c] = np.where(df[c] < 0, np.nan, df[c])

df = df.drop(columns=[c for c in date_cols if c in df.columns], errors='ignore')

X = df.drop(columns=['target_binary'])
y = df['target_binary']

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

print(f"[*] Clinical Dataset Evaluated: N = {len(X)} observations, M = {X.shape[1]} raw features.")

# ==============================================================================
# 2. PIPELINE DEFINITIONS & UTILS
# ==============================================================================
def sanitize_feature_names(df_in):
    """Replaces JSON-invalid characters to prevent LightGBM crashes."""
    if isinstance(df_in, pd.DataFrame):
        df_in.columns = [re.sub(r'[^a-zA-Z0-9_]', '_', str(c)).strip('_') for c in df_in.columns]
        cols = pd.Series(df_in.columns)
        for dup in cols[cols.duplicated()].unique():
            cols[cols[cols == dup].index.values.tolist()] = [dup + '_' + str(i) if i != 0 else dup for i in range(sum(cols == dup))]
        df_in.columns = cols
    return df_in

class CorrelationDropper(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.85):
        self.threshold = threshold
        self.to_drop_ = []

    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.to_drop_ = [column for column in upper.columns if any(upper[column] > self.threshold)]
        return self

    def transform(self, X, y=None):
        return X.drop(columns=self.to_drop_, errors='ignore')

def get_classic_preprocessor(imp_name, current_seed):
    if imp_name == 'MICE': imp = IterativeImputer(random_state=current_seed, max_iter=10)
    elif imp_name == 'KNN': imp = KNNImputer(n_neighbors=5)
    else: imp = SimpleImputer(strategy='median')

    num_pipe = Pipeline([('imp', imp), ('scaler', StandardScaler())])
    cat_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

def get_native_ood(X_tr, X_te):
    X_tr_nat = X_tr.copy()
    X_te_nat = X_te.copy()

    X_tr_nat[cat_cols] = X_tr_nat[cat_cols].fillna('Unknown')
    X_te_nat[cat_cols] = X_te_nat[cat_cols].fillna('Unknown')

    medians = X_tr_nat[num_cols].median()
    X_tr_nat[num_cols] = X_tr_nat[num_cols].fillna(medians)
    X_te_nat[num_cols] = X_te_nat[num_cols].fillna(medians)

    d_cols = [c for c in num_cols if 'delta' in c]
    X_tr_nat[d_cols] = X_tr_nat[d_cols].fillna(-999)
    X_te_nat[d_cols] = X_te_nat[d_cols].fillna(-999)

    for c in cat_cols:
        known = list(X_tr_nat[c].unique())
        X_te_nat[c] = X_te_nat[c].apply(lambda x: x if x in known else 'Unknown')
        ctype = pd.CategoricalDtype(categories=list(set(known + ['Unknown'])), ordered=False)
        X_tr_nat[c] = X_tr_nat[c].astype(ctype)
        X_te_nat[c] = X_te_nat[c].astype(ctype)

    return sanitize_feature_names(X_tr_nat), sanitize_feature_names(X_te_nat)

def get_metrics(y_t, y_p, y_prob, prefix=''):
    tn, fp, fn, tp = confusion_matrix(y_t, y_p, labels=[0,1]).ravel()
    return {
        f'{prefix}AUC': roc_auc_score(y_t, y_prob) if y_prob is not None else np.nan,
        f'{prefix}F1': f1_score(y_t, y_p) if np.sum(y_p) > 0 else 0.0,
        f'{prefix}Sensitivity': tp / (tp + fn + 1e-9),
        f'{prefix}Specificity': tn / (tn + fp + 1e-9)
    }

# ==============================================================================
# 3. OPTIMIZATION ENGINE
# ==============================================================================
def optimize_algorithm(model_name, imp_name, X_tr, y_tr, current_seed, is_native=False):
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=current_seed)

    if not is_native:
        pipeline_prep = Pipeline([
            ('prep', get_classic_preprocessor(imp_name, current_seed)),
            ('corr_drop', CorrelationDropper(threshold=0.85))
        ])
        X_opt = pipeline_prep.fit_transform(X_tr)
        X_opt = sanitize_feature_names(X_opt)
    else:
        X_opt = X_tr

    def objective(trial):
        try:
            if is_native:
                if model_name == 'CATB_Native':
                    clf = CatBoostClassifier(iterations=trial.suggest_categorical('iterations', [30, 50]),
                                             depth=trial.suggest_categorical('depth', [2, 3]),
                                             learning_rate=trial.suggest_categorical('learning_rate', [0.01, 0.05]),
                                             l2_leaf_reg=trial.suggest_categorical('l2_leaf_reg', [1.0, 3.0, 5.0]),
                                             random_state=current_seed, verbose=False, cat_features=cat_cols,
                                             auto_class_weights='Balanced')
                elif model_name == 'LGBM_Native':
                    clf = LGBMClassifier(n_estimators=trial.suggest_categorical('n_estimators', [30, 50]),
                                         max_depth=trial.suggest_categorical('max_depth', [2, 3]),
                                         learning_rate=trial.suggest_categorical('learning_rate', [0.01, 0.05]),
                                         num_leaves=trial.suggest_categorical('num_leaves', [4, 7]),
                                         min_child_samples=trial.suggest_categorical('min_child_samples', [2, 5]),
                                         verbose=-1, random_state=current_seed, is_unbalance=True)
                elif model_name == 'XGB_Native':
                    clf = XGBClassifier(n_estimators=trial.suggest_categorical('n_estimators', [30, 50]),
                                        max_depth=trial.suggest_categorical('max_depth', [2, 3]),
                                        learning_rate=trial.suggest_categorical('learning_rate', [0.01, 0.05]),
                                        subsample=trial.suggest_categorical('subsample', [0.8, 1.0]),
                                        colsample_bytree=trial.suggest_categorical('colsample_bytree', [0.8, 1.0]),
                                        enable_categorical=True, tree_method='hist', random_state=current_seed)
                scores = []
                for tr_idx, val_idx in cv_inner.split(X_opt, y_tr):
                    X_f_tr, X_f_val = X_opt.iloc[tr_idx], X_opt.iloc[val_idx]
                    y_f_tr, y_f_val = y_tr.iloc[tr_idx], y_tr.iloc[val_idx]
                    with HiddenPrints():
                        clf.fit(X_f_tr, y_f_tr)
                    scores.append(roc_auc_score(y_f_val, clf.predict_proba(X_f_val)[:, 1]))
                return np.mean(scores)

            else:
                if model_name == 'LR':
                    clf = LogisticRegression(penalty=trial.suggest_categorical('penalty', ['l1', 'l2']),
                                             C=trial.suggest_categorical('C', [0.1, 1.0, 5.0]),
                                             solver='liblinear', class_weight='balanced', random_state=current_seed)
                elif model_name == 'RF':
                    clf = RandomForestClassifier(n_estimators=trial.suggest_categorical('n_estimators', [30, 50]),
                                                 max_depth=trial.suggest_categorical('max_depth', [2, 3]),
                                                 min_samples_split=trial.suggest_categorical('min_samples_split', [2, 5]),
                                                 class_weight='balanced', random_state=current_seed)
                elif model_name == 'SVM':
                    clf = SVC(C=trial.suggest_categorical('C', [0.1, 1.0, 5.0]),
                              kernel=trial.suggest_categorical('kernel', ['linear', 'rbf']),
                              probability=True, class_weight='balanced', random_state=current_seed)
                elif model_name == 'XGB':
                    clf = XGBClassifier(n_estimators=trial.suggest_categorical('n_estimators', [30, 50]),
                                        max_depth=trial.suggest_categorical('max_depth', [2, 3]),
                                        learning_rate=trial.suggest_categorical('learning_rate', [0.01, 0.05]),
                                        subsample=trial.suggest_categorical('subsample', [0.8, 1.0]),
                                        eval_metric='logloss', random_state=current_seed)
                elif model_name == 'LGBM':
                    clf = LGBMClassifier(n_estimators=trial.suggest_categorical('n_estimators', [30, 50]),
                                         max_depth=trial.suggest_categorical('max_depth', [2, 3]),
                                         learning_rate=trial.suggest_categorical('learning_rate', [0.01, 0.05]),
                                         num_leaves=trial.suggest_categorical('num_leaves', [4, 7]),
                                         min_child_samples=trial.suggest_categorical('min_child_samples', [2, 5]),
                                         verbose=-1, is_unbalance=True, random_state=current_seed)
                elif model_name == 'CATB':
                    clf = CatBoostClassifier(iterations=trial.suggest_categorical('iterations', [30, 50]),
                                             depth=trial.suggest_categorical('depth', [2, 3]),
                                             learning_rate=trial.suggest_categorical('learning_rate', [0.01, 0.05]),
                                             l2_leaf_reg=trial.suggest_categorical('l2_leaf_reg', [1.0, 3.0, 5.0]),
                                             random_state=current_seed, verbose=False, auto_class_weights='Balanced')

                rf_selector = SelectFromModel(RandomForestClassifier(n_estimators=50, max_depth=3, class_weight='balanced', random_state=current_seed), threshold=-np.inf, max_features=15)
                pipe_eval = Pipeline([
                    ('feat_sel', rf_selector),
                    ('clf', clf)
                ])
                with HiddenPrints():
                    return np.mean(cross_val_score(pipe_eval, X_opt, y_tr, cv=cv_inner, scoring='roc_auc'))

        except Exception:
            return 0.5

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=current_seed))
    study.optimize(objective, n_trials=20)

    try:
        return study.best_params, study.best_value
    except ValueError:
        return None, None

# ==============================================================================
# 4. EXECUTION LOOP
# ==============================================================================
experiments_classic = [
    ('LR', 'Median'), ('LR', 'KNN'), ('LR', 'MICE'),
    ('RF', 'Median'), ('RF', 'KNN'), ('RF', 'MICE'),
    ('SVM', 'Median'), ('SVM', 'KNN'), ('SVM', 'MICE'),
    ('XGB', 'Median'), ('XGB', 'KNN'), ('XGB', 'MICE'),
    ('LGBM', 'Median'), ('LGBM', 'KNN'), ('LGBM', 'MICE'),
    ('CATB', 'Median'), ('CATB', 'KNN'), ('CATB', 'MICE')
]

experiments_native = [
    ('CATB_Native', 'Native_OOD'),
    ('LGBM_Native', 'Native_OOD'),
    ('XGB_Native', 'Native_OOD')
]

batch_results = []
trained_models_batch = {}
features_selected_batch = []

print(f"\nCommencing nested cross-validation and hyperparameter optimization for seeds {START_SEED} through {END_SEED-1}.")

for current_seed in tqdm(range(START_SEED, END_SEED), desc="Processing Validation Splits"):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=current_seed, stratify=y)

    missing_cols = X_train.columns[X_train.isnull().any()].tolist()
    for col in missing_cols:
        X_train[f"{col}_missing_flag"] = X_train[col].isnull().astype(int)
        X_test[f"{col}_missing_flag"] = X_test[col].isnull().astype(int)

    X_train_nat, X_test_nat = get_native_ood(X_train, X_test)

    # 4.1 Classic Branch
    for model_name, imp_name in experiments_classic:
        params, cv_auc = optimize_algorithm(model_name, imp_name, X_train, y_train, is_native=False, current_seed=current_seed)

        if not params:
            continue

        pipe_prep = Pipeline([('prep', get_classic_preprocessor(imp_name, current_seed)), ('corr_drop', CorrelationDropper(threshold=0.85))])
        X_tr_p = pipe_prep.fit_transform(X_train)
        X_te_p = pipe_prep.transform(X_test)

        X_tr_p = sanitize_feature_names(X_tr_p)
        X_te_p = sanitize_feature_names(X_te_p)

        selector = SelectFromModel(RandomForestClassifier(n_estimators=50, max_depth=3, class_weight='balanced', random_state=current_seed), max_features=15, threshold=-np.inf)
        X_tr_f = selector.fit_transform(X_tr_p, y_train)
        X_te_f = selector.transform(X_te_p)

        try:
            selected_mask = selector.get_support()
            selected_features = X_tr_p.columns[selected_mask].tolist()
            for feat in selected_features:
                features_selected_batch.append({'Seed': current_seed, 'Algorithm': model_name, 'Imputation': imp_name, 'Feature': feat})
        except Exception:
            pass

        if model_name == 'LR': clf = LogisticRegression(**params, solver='liblinear', class_weight='balanced', random_state=current_seed)
        elif model_name == 'RF': clf = RandomForestClassifier(**params, class_weight='balanced', random_state=current_seed)
        elif model_name == 'SVM': clf = SVC(**params, probability=True, class_weight='balanced', random_state=current_seed)
        elif model_name == 'XGB': clf = XGBClassifier(**params, eval_metric='logloss', random_state=current_seed)
        elif model_name == 'LGBM': clf = LGBMClassifier(**params, verbose=-1, is_unbalance=True, random_state=current_seed)
        elif model_name == 'CATB': clf = CatBoostClassifier(**params, random_state=current_seed, auto_class_weights='Balanced', verbose=False)

        try:
            with HiddenPrints(): clf.fit(X_tr_f, y_train)
            y_pred_te = clf.predict(X_te_f)
            y_prob_te = clf.predict_proba(X_te_f)[:, 1]
            y_pred_tr = clf.predict(X_tr_f)
            y_prob_tr = clf.predict_proba(X_tr_f)[:, 1]

            row = {
                'Seed': current_seed, 'Algorithm': model_name, 'Imputation': imp_name,
                'CV_AUC_Score': cv_auc, 'Hyperparameters_JSON': json.dumps(params)
            }
            row.update(get_metrics(y_train, y_pred_tr, y_prob_tr, 'Train_'))
            row.update(get_metrics(y_test, y_pred_te, y_prob_te, 'Test_'))
            batch_results.append(row)

            trained_models_batch[f"{model_name}_{imp_name}_seed_{current_seed}"] = {
                'model': clf, 'X_train_proc': X_tr_f, 'X_test_proc': X_te_f, 'AUC': row['Test_AUC']
            }
        except Exception:
            pass

    # 4.2 Native Branch
    for model_name, imp_name in experiments_native:
        params, cv_auc = optimize_algorithm(model_name, imp_name, X_train_nat, y_train, is_native=True, current_seed=current_seed)

        if not params:
            continue

        if model_name == 'CATB_Native': clf = CatBoostClassifier(**params, random_state=current_seed, verbose=False, cat_features=cat_cols, auto_class_weights='Balanced')
        elif model_name == 'LGBM_Native': clf = LGBMClassifier(**params, verbose=-1, random_state=current_seed, is_unbalance=True)
        elif model_name == 'XGB_Native': clf = XGBClassifier(**params, enable_categorical=True, tree_method='hist', random_state=current_seed)

        try:
            with HiddenPrints(): clf.fit(X_train_nat, y_train)

            y_pred_te = clf.predict(X_test_nat)
            y_prob_te = clf.predict_proba(X_test_nat)[:, 1]
            y_pred_tr = clf.predict(X_train_nat)
            y_prob_tr = clf.predict_proba(X_train_nat)[:, 1]

            row = {
                'Seed': current_seed, 'Algorithm': model_name, 'Imputation': imp_name,
                'CV_AUC_Score': cv_auc, 'Hyperparameters_JSON': json.dumps(params)
            }
            row.update(get_metrics(y_train, y_pred_tr, y_prob_tr, 'Train_'))
            row.update(get_metrics(y_test, y_pred_te, y_prob_te, 'Test_'))
            batch_results.append(row)

            trained_models_batch[f"{model_name}_{imp_name}_seed_{current_seed}"] = {
                'model': clf, 'X_train_proc': X_train_nat, 'X_test_proc': X_test_nat, 'AUC': row['Test_AUC']
            }
        except Exception:
            pass

# ==============================================================================
# 5. DATA ACCUMULATION AND EXPORT
# ==============================================================================
df_batch = pd.DataFrame(batch_results)

if os.path.exists(MASTER_CSV_PATH):
    print(f"\n[*] Existing master log identified. Appending results...")
    df_existing = pd.read_csv(MASTER_CSV_PATH, sep=';', decimal=',')
    df_master = pd.concat([df_existing, df_batch], ignore_index=True)
else:
    print(f"\n[*] Creating new master log file...")
    df_master = df_batch.copy()

if not df_master.empty:
    df_master = df_master.drop_duplicates(subset=['Seed', 'Algorithm', 'Imputation'], keep='last')
    df_master.to_csv(MASTER_CSV_PATH, index=False, sep=';', decimal=',')

df_feat_batch = pd.DataFrame(features_selected_batch)
if os.path.exists(FEATURES_LOG_PATH) and not df_feat_batch.empty:
    df_feat_existing = pd.read_csv(FEATURES_LOG_PATH, sep=';')
    df_feat_master = pd.concat([df_feat_existing, df_feat_batch], ignore_index=True)
    df_feat_master = df_feat_master.drop_duplicates(subset=['Seed', 'Algorithm', 'Imputation', 'Feature'], keep='last')
    df_feat_master.to_csv(FEATURES_LOG_PATH, index=False, sep=';')
elif not df_feat_batch.empty:
    df_feat_master = df_feat_batch.copy()
    df_feat_master.to_csv(FEATURES_LOG_PATH, index=False, sep=';')

if not df_master.empty:
    df_master['Model_Key'] = df_master['Algorithm'] + "_" + df_master['Imputation']
    stats_summary = df_master.groupby('Model_Key')[['Test_AUC', 'Test_F1', 'Test_Sensitivity', 'Test_Specificity']].agg(['mean', 'std'])

    current_total_seeds = df_master['Seed'].nunique()
    if current_total_seeds >= 5:
        for metric in ['Test_AUC', 'Test_F1', 'Test_Sensitivity', 'Test_Specificity']:
            stats_summary[(metric, 'CI_Lower_95')] = df_master.groupby('Model_Key')[metric].quantile(0.025)
            stats_summary[(metric, 'CI_Upper_95')] = df_master.groupby('Model_Key')[metric].quantile(0.975)

    stats_summary = stats_summary.sort_values(by=('Test_AUC', 'mean'), ascending=False)
    stats_summary.to_csv(f"{OUTPUT_DIR}/Statistical_Summary_By_Algorithm.csv")

# ==============================================================================
# 6. METHODOLOGICAL VISUALIZATIONS
# ==============================================================================
print("\nGenerating Methodological Visualizations...")

if not df_master.empty:
    fig, ax = plt.subplots(figsize=(14, 8))
    order = stats_summary.index.tolist()
    sns.boxplot(x='Model_Key', y='Test_AUC', data=df_master, order=order, color='#EAEAEA', showfliers=False, ax=ax)
    sns.stripplot(x='Model_Key', y='Test_AUC', data=df_master, order=order, alpha=0.5, jitter=True, size=4, palette='viridis', ax=ax)

    ax.set_title(f"Algorithm Stability Across {current_total_seeds} Partitions", fontsize=14, fontweight='bold')
    ax.set_ylabel("Test Set AUC-ROC", fontsize=12)
    ax.set_xlabel("Algorithm & Imputation Strategy", fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.axhline(0.5, color='red', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots_models/Fig1_Algorithm_Stability.png", dpi=300)
    plt.close()

    if current_total_seeds > 2:
        top_3_models = stats_summary.index[:3].tolist()
        fig, ax = plt.subplots(figsize=(9, 6))

        colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
        for idx, model_key in enumerate(top_3_models):
            subset = df_master[df_master['Model_Key'] == model_key]
            sns.kdeplot(data=subset, x='Test_AUC', linewidth=2, label=f"{model_key} (Mean: {subset['Test_AUC'].mean():.3f})", color=colors[idx], ax=ax)

        ax.set_title(f"Empirical Density Comparison of Top 3 Models (N_Seeds={current_total_seeds})", fontsize=12, fontweight='bold')
        ax.set_xlabel("Test Set AUC-ROC", fontsize=11)
        ax.set_ylabel("Density", fontsize=11)
        ax.legend(title="Top Models", loc='upper left')
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/plots_models/Fig2_Top3_Density_Comparison.png", dpi=300)
        plt.close()

    heatmap_data = stats_summary.xs('mean', axis=1, level=1)[['Test_AUC', 'Test_F1', 'Test_Sensitivity', 'Test_Specificity']]
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap='YlGnBu', cbar_kws={'label': 'Mean Score'}, ax=ax)
    ax.set_title(f"Average Performance Metrics Across {current_total_seeds} Seeds", fontsize=12, fontweight='bold')
    ax.set_ylabel("Algorithm")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/plots_models/Fig3_Metrics_Heatmap.png", dpi=300)
    plt.close()

    if 'df_feat_master' in locals() and not df_feat_master.empty:
        feat_freq = df_feat_master['Feature'].value_counts()
        total_classic_runs = df_master[~df_master['Algorithm'].str.contains('Native')].shape[0]

        df_feat_freq = pd.DataFrame({
            'Feature': feat_freq.index,
            'Selection_Percentage': (feat_freq.values / total_classic_runs) * 100
        }).head(20)

        fig, ax = plt.subplots(figsize=(10, 8))
        sns.barplot(x='Selection_Percentage', y='Feature', data=df_feat_freq, palette='mako', ax=ax)
        ax.set_title(f"Feature Selection Stability (Top 20 Features)\nEvaluated across {total_classic_runs} classic pipeline fits", fontsize=12, fontweight='bold')
        ax.set_xlabel("Selection Frequency (%)", fontsize=11)
        ax.set_ylabel("")
        ax.set_xlim(0, 105)
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/plots_models/Fig4_Feature_Stability.png", dpi=300)
        plt.close()

# ==============================================================================
# POST-EXECUTION ARTIFACT GENERATION FOR MANUSCRIPT
# ==============================================================================
PLOTS_DIR = f"{OUTPUT_DIR}/Manuscript_Artifacts"
os.makedirs(PLOTS_DIR, exist_ok=True)

# Format definitions for publication submission
IMAGE_FORMATS = ['png', 'svg']
DPI_SETTING = 300

print("="*80)
print("EXTRACTING PUBLICATION DATA AND GENERATING PLOTS")
print("="*80)

# ==============================================================================
# 1. CLINICAL COHORT ANALYSIS (EMPIRICAL STRESS TEST)
# ==============================================================================
if os.path.exists(MASTER_CSV_PATH):
    try:
        df_master_eval = pd.read_csv(MASTER_CSV_PATH, sep=';', decimal=',')
    except Exception:
        df_master_eval = pd.read_csv(MASTER_CSV_PATH)

    df_master_eval['Model_Key'] = df_master_eval['Algorithm'] + " (" + df_master_eval['Imputation'] + ")"
    total_seeds = df_master_eval['Seed'].nunique()

    # Statistical Summary Compilation
    stats_summary_eval = df_master_eval.groupby('Model_Key')[['Test_AUC', 'Test_F1', 'Test_Sensitivity', 'Test_Specificity']].agg(['mean', 'std', 'min', 'max'])

    # Calculate 95% Confidence Intervals
    for metric in ['Test_AUC', 'Test_F1', 'Test_Sensitivity', 'Test_Specificity']:
        stats_summary_eval[(metric, 'CI_Lower_95')] = df_master_eval.groupby('Model_Key')[metric].quantile(0.025)
        stats_summary_eval[(metric, 'CI_Upper_95')] = df_master_eval.groupby('Model_Key')[metric].quantile(0.975)

    stats_summary_eval = stats_summary_eval.sort_values(by=('Test_AUC', 'mean'), ascending=False)

    # Format the Official Manuscript Table
    formatted_table = pd.DataFrame()
    formatted_table['Algorithm (Imputation)'] = stats_summary_eval.index
    formatted_table['Mean AUC (95% CI)'] = stats_summary_eval.apply(lambda row: f"{row[('Test_AUC', 'mean')]:.3f} ({row[('Test_AUC', 'CI_Lower_95')]:.3f} - {row[('Test_AUC', 'CI_Upper_95')]:.3f})", axis=1).values
    formatted_table['Mean F1 (95% CI)'] = stats_summary_eval.apply(lambda row: f"{row[('Test_F1', 'mean')]:.3f} ({row[('Test_F1', 'CI_Lower_95')]:.3f} - {row[('Test_F1', 'CI_Upper_95')]:.3f})", axis=1).values
    formatted_table['Mean Sens (95% CI)'] = stats_summary_eval.apply(lambda row: f"{row[('Test_Sensitivity', 'mean')]:.3f} ({row[('Test_Sensitivity', 'CI_Lower_95')]:.3f} - {row[('Test_Sensitivity', 'CI_Upper_95')]:.3f})", axis=1).values
    formatted_table['Mean Spec (95% CI)'] = stats_summary_eval.apply(lambda row: f"{row[('Test_Specificity', 'mean')]:.3f} ({row[('Test_Specificity', 'CI_Lower_95')]:.3f} - {row[('Test_Specificity', 'CI_Upper_95')]:.3f})", axis=1).values

    formatted_table.to_csv(f"{PLOTS_DIR}/Table_Manuscript_Main_Results.csv", index=False)
    print(" -> Official Manuscript Table (with 95% CIs) generated.")

    # Data for textual insertion (We focus on the Top models overall)
    best_algo_key = stats_summary_eval.index[0]
    worst_algo_key = stats_summary_eval.index[-1]

    overall_mean_auc = df_master_eval['Test_AUC'].mean()
    overall_min_sens = df_master_eval['Test_Sensitivity'].min()
    overall_max_sens = df_master_eval['Test_Sensitivity'].max()
    overall_mean_sens = df_master_eval['Test_Sensitivity'].mean()
    overall_std_sens = df_master_eval['Test_Sensitivity'].std()

    # --------------------------------------------------------------------------
    # Plot 1: Overall AUC Density (Aggregated across all top pipelines to match manuscript)
    # --------------------------------------------------------------------------
    top_5_models = stats_summary_eval.index[:5].tolist()
    df_top5 = df_master_eval[df_master_eval['Model_Key'].isin(top_5_models)]

    overall_ci_low = df_top5['Test_AUC'].quantile(0.025)
    overall_ci_high = df_top5['Test_AUC'].quantile(0.975)
    overall_mean = df_top5['Test_AUC'].mean()

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.kdeplot(df_top5['Test_AUC'], color='#2c7bb6', linewidth=2, label=f"Mean AUC ({overall_mean:.3f})", ax=ax)

    if len(ax.lines) > 0:
        kdeline = ax.lines[0]
        xs = kdeline.get_xdata()
        ys = kdeline.get_ydata()
        ax.fill_between(xs, 0, ys, where=(xs >= overall_ci_low) & (xs <= overall_ci_high), facecolor='#2c7bb6', alpha=0.3, label=f"95% CI")

    ax.set_title("Empirical Density Distribution of the Optimized Pipeline's AUC-ROC", fontsize=11, pad=12)
    ax.set_xlabel("Holdout Set AUC-ROC", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.legend(loc='upper left', fontsize=9, frameon=False)
    sns.despine()
    plt.tight_layout()

    for fmt in IMAGE_FORMATS:
        plt.savefig(f"{PLOTS_DIR}/Plot_Empirical_AUC_Density.{fmt}", dpi=DPI_SETTING, bbox_inches='tight')
    plt.close()

    # --------------------------------------------------------------------------
    # Plot 2: Comprehensive Metrics Distribution (Boxplot for Top Models)
    # --------------------------------------------------------------------------
    metrics_melted = df_top5.melt(id_vars=['Seed'], value_vars=['Test_AUC', 'Test_F1', 'Test_Sensitivity', 'Test_Specificity'],
                                    var_name='Metric', value_name='Score')
    metrics_melted['Metric'] = metrics_melted['Metric'].str.replace('Test_', '')

    fig, ax = plt.subplots(figsize=(9, 5))
    sns.boxplot(x='Metric', y='Score', data=metrics_melted, color='#EFEFEF', width=0.4, showfliers=False, ax=ax)
    sns.stripplot(x='Metric', y='Score', data=metrics_melted, alpha=0.3, jitter=True, size=4, color='#404040', ax=ax)

    ax.set_title("Distribution of Classification Metrics (Top Performing Configurations)", fontsize=11, pad=12)
    ax.set_ylabel("Score", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylim(-0.05, 1.05)
    plt.axhline(0.5, color='gray', linestyle=':', alpha=0.7)
    sns.despine()
    plt.tight_layout()

    for fmt in IMAGE_FORMATS:
        plt.savefig(f"{PLOTS_DIR}/Plot_Global_Metrics_Distribution.{fmt}", dpi=DPI_SETTING, bbox_inches='tight')
    plt.close()
    print(" -> Clinical plots (Density and Boxplots) generated.")

else:
    print("[!] ERROR: Master Log not found.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 3.0 MB/s eta 0:00:00
INITIALIZING MULTI-ALGORITHM STRESS TEST (BATCH: SEEDS 100 TO 100)
[*] Clinical Dataset Evaluated: N = 75 observations, M = 17 raw features.

Commencing nested cross-validation and hyperparameter optimization for seeds 100 through 100.


Processing Validation Splits:   0%|          | 0/1 [00:00<?, ?it/s]


[*] Creating new master log file...

Generating Methodological Visualizations...
EXTRACTING PUBLICATION DATA AND GENERATING PLOTS
 -> Official Manuscript Table (with 95% CIs) generated.
 -> Clinical plots (Density and Boxplots) generated.


In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP & DEPENDENCIES
# ==============================================================================
!pip install xgboost shap openpyxl -q

import os
import sys
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Compatibility Patches for Modern Scikit-Learn
import sklearn.utils._array_api
if not hasattr(sklearn.utils._array_api, '_logsumexp'):
    from scipy.special import logsumexp
    sklearn.utils._array_api._logsumexp = logsumexp
if not hasattr(sklearn.utils._array_api, '_half_multinomial_loss'):
    sklearn.utils._array_api._half_multinomial_loss = lambda *args, **kwargs: 0.0

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score

# Configure Scikit-Learn to output DataFrames
sklearn.set_config(transform_output="pandas")
warnings.filterwarnings('ignore')
plt.style.use('default')

# Configuration
NUM_SEEDS = 100
OUTPUT_DIR = '/content/drive/MyDrive/avc/seed/outputs/simulation'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/plots", exist_ok=True)

# ==============================================================================
# IN SILICO VARIANCE FUNNEL SIMULATION (Section 2.3 of the Manuscript)
# ==============================================================================
print("Generating In Silico Variance Funnel Simulation...")

N_UNIVERSE = 10000
X_univ, y_univ = make_classification(
    n_samples=N_UNIVERSE,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_clusters_per_class=2,
    flip_y=0.15,  # Ajustado para 0.15 para corresponder exatamente ao manuscrito
    weights=[0.8, 0.2],
    class_sep=0.8,
    random_state=42
)

sample_sizes = range(50, 500, 25)
sim_results = []
sim_clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)

for N in tqdm(sample_sizes, desc="Simulating varying sample sizes"):
    for s in range(NUM_SEEDS):
        X_samp, _, y_samp, _ = train_test_split(X_univ, y_univ, train_size=N, random_state=s, stratify=y_univ)
        X_tr, X_te, y_tr, y_te = train_test_split(X_samp, y_samp, test_size=0.20, random_state=s, stratify=y_samp)

        sim_clf.fit(X_tr, y_tr)
        auc = roc_auc_score(y_te, sim_clf.predict_proba(X_te)[:, 1])
        sim_results.append({'N_Size': N, 'Seed': s, 'Test_AUC': auc})

df_sim = pd.DataFrame(sim_results)
df_sim.to_csv(f"{OUTPUT_DIR}/Simulation_Results.csv", index=False)

# ==============================================================================
# PLOT GENERATION: VARIANCE FUNNEL
# ==============================================================================
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(x='N_Size', y='Test_AUC', data=df_sim, color='#E0E0E0', ax=ax, showfliers=False)
sns.stripplot(x='N_Size', y='Test_AUC', data=df_sim, color='#2C3E50', alpha=0.5, jitter=True, size=4, ax=ax)

global_mean = df_sim['Test_AUC'].mean()
plt.axhline(global_mean, color='red', linestyle='--', alpha=0.7, label=f'Global Mean AUC ({global_mean:.3f})')
ax.set_xlabel("Cohort Sample Size (N)")
ax.set_ylabel("Test Set AUC-ROC")
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/plots/Variance_Funnel.png", dpi=300)
plt.close()

df_summary = df_sim.groupby('N_Size')['Test_AUC'].agg(['mean', 'std', 'min', 'max'])
df_summary['Range'] = df_summary['max'] - df_summary['min']
df_summary.to_csv(f"{OUTPUT_DIR}/Simulation_Summary.csv")

print(f"Simulation completed. Summary and plots exported to {OUTPUT_DIR}")

Generating In Silico Variance Funnel Simulation...


Simulating varying sample sizes:   0%|          | 0/18 [00:00<?, ?it/s]

Simulation completed. Summary and plots exported to /content/drive/MyDrive/avc/seed/outputs/simulation


In [ ]:
# ==============================================================================
# SCRIPT FOR GENERATING MANUSCRIPT TABLE S1 (TRANSLATED BASELINE CHARACTERISTICS)
# ==============================================================================
import pandas as pd
import numpy as np
import os
import re

# Directory definitions
FILE_PATH = '/content/drive/MyDrive/avc/seed/avci.xlsx'
OUTPUT_DIR = '/content/drive/MyDrive/avc/seed/outputs/table_s1'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nGenerating Translated Manuscript Supplementary Table S1...")

# 1. Load and sanitize dataset columns
df_raw = pd.read_excel(FILE_PATH)
df_raw.columns = [re.sub(r'[^a-z0-9_]+', '_', str(c).lower()).strip('_') for c in df_raw.columns]

# Target variable formatting
target_col = 'escala_de_rankin_modificada_em_90_dias'
df_raw = df_raw.dropna(subset=[target_col])
df_raw['target_binary'] = np.where(df_raw[target_col] <= 3, 0, 1)

# Isolate the final cohort (N=75) by removing ghost patients (missingness > 50%)
cols_to_ignore = ['record_id', target_col, 'target_binary']
feature_cols = [c for c in df_raw.columns if c not in cols_to_ignore]
missing_pct = df_raw[feature_cols].isnull().mean(axis=1)
df_cohort = df_raw[missing_pct <= 0.50].copy()

# Force designated numerical columns to be float numeric to prevent summation errors
df_cohort['idade'] = pd.to_numeric(df_cohort['idade'], errors='coerce')

# Split cohort by functional outcome
df_fav = df_cohort[df_cohort['target_binary'] == 0]
df_unfav = df_cohort[df_cohort['target_binary'] == 1]

n_overall = len(df_cohort)
n_fav = len(df_fav)
n_unfav = len(df_unfav)

print(f"[*] Final Cohort Size: N = {n_overall} (Favorable: {n_fav}, Unfavorable: {n_unfav})")

# Dictionary to dynamically map Portuguese categorical values to English equivalents
translation_dict = {
    'sexo_biol_gico_atribu_do_ao_nascer': {
        'masculino': 'Male',
        'feminino': 'Female'
    },
    'ra_a_cor_autodeclarada': {
        'branca': 'White',
        'preta': 'Black',
        'parda': 'Mixed/Pardo'
    },
    'escolaridade': {
        'analfabeto': 'Illiterate',
        'fundamental completo': 'Primary school completed',
        'fundamental incompleto': 'Primary school uncompleted',
        'médio completo': 'High school completed',
        'médio incompleto': 'High school uncompleted',
        'superior completo': 'University degree completed'
    }
}

# 2. Define baseline variables for Table S1
variables_to_profile = {
    'idade': ('Age (years)', 'numeric'),
    'sexo_biol_gico_atribu_do_ao_nascer': ('Biological Sex', 'categorical'),
    'ra_a_cor_autodeclarada': ('Self-declared Race/Color', 'categorical'),
    'escolaridade': ('Education Level', 'categorical')
}

table_rows = []

# Header Row
table_rows.append({
    'Characteristic': 'Total Patients',
    'Overall': f"{n_overall} (100%)",
    'Favorable Outcome (mRS <= 3)': f"{n_fav} ({ (n_fav/n_overall)*100:.1f}%)",
    'Unfavorable Outcome (mRS > 3)': f"{n_unfav} ({ (n_unfav/n_overall)*100:.1f}%)"
})

# Compile statistics dynamically using strict key matching
for col, (label, dtype) in variables_to_profile.items():
    if col not in df_cohort.columns:
        print(f"[!] Warning: Feature '{col}' not found in the dataset. Skipping...")
        continue

    actual_col = col

    if dtype == 'numeric':
        mean_ov, std_ov = df_cohort[actual_col].mean(), df_cohort[actual_col].std()
        mean_f, std_f = df_fav[actual_col].mean(), df_fav[actual_col].std()
        mean_u, std_u = df_unfav[actual_col].mean(), df_unfav[actual_col].std()

        table_rows.append({
            'Characteristic': f"{label}, Mean (SD)",
            'Overall': f"{mean_ov:.1f} ({std_ov:.1f})",
            'Favorable Outcome (mRS <= 3)': f"{mean_f:.1f} ({std_f:.1f})",
            'Unfavorable Outcome (mRS > 3)': f"{mean_u:.1f} ({std_u:.1f})"
        })

        # Calculate missingness
        missing_ov = df_cohort[actual_col].isnull().sum()
        if missing_ov > 0:
            missing_f = df_fav[actual_col].isnull().sum()
            missing_u = df_unfav[actual_col].isnull().sum()
            table_rows.append({
                'Characteristic': f"  - Missing, N (%)",
                'Overall': f"{missing_ov} ({(missing_ov/n_overall)*100:.1f}%)",
                'Favorable Outcome (mRS <= 3)': f"{missing_f} ({(missing_f/n_fav)*100:.1f}%)",
                'Unfavorable Outcome (mRS > 3)': f"{missing_u} ({(missing_u/n_unfav)*100:.1f}%)"
            })

    elif dtype == 'categorical':
        table_rows.append({
            'Characteristic': f"{label}, N (%)",
            'Overall': "", 'Favorable Outcome (mRS <= 3)': "", 'Unfavorable Outcome (mRS > 3)': ""
        })

        categories = df_cohort[actual_col].dropna().unique()
        for cat in categories:
            cat_str_lower = str(cat).lower().strip()

            # Translate raw Portuguese terms to English equivalents
            translated_cat = cat
            if col in translation_dict and cat_str_lower in translation_dict[col]:
                translated_cat = translation_dict[col][cat_str_lower]
            else:
                translated_cat = str(cat).capitalize()

            table_rows.append({
                'Characteristic': f"  - {translated_cat}",
                'Overall': f"{df_cohort[df_cohort[actual_col] == cat].shape[0]} ({(df_cohort[df_cohort[actual_col] == cat].shape[0]/n_overall)*100:.1f}%)",
                'Favorable Outcome (mRS <= 3)': f"{df_fav[df_fav[actual_col] == cat].shape[0]} ({(df_fav[df_fav[actual_col] == cat].shape[0]/n_fav)*100:.1f}%)",
                'Unfavorable Outcome (mRS > 3)': f"{df_unfav[df_unfav[actual_col] == cat].shape[0]} ({(df_unfav[df_unfav[actual_col] == cat].shape[0]/n_unfav)*100:.1f}%)"
            })

        # Calculate missingness
        missing_ov = df_raw[col].isnull().sum()
        if missing_ov > 0:
            missing_f = df_fav[col].isnull().sum()
            missing_u = df_unfav[col].isnull().sum()
            table_rows.append({
                'Characteristic': f"  - Missing",
                'Overall': f"{missing_ov} ({(missing_ov/n_overall)*100:.1f}%)",
                'Favorable Outcome (mRS <= 3)': f"{missing_f} ({(missing_f/n_fav)*100:.1f}%)",
                'Unfavorable Outcome (mRS > 3)': f"{missing_u} ({(missing_u/n_unfav)*100:.1f}%)"
            })

# 3. Export to Excel and CSV
df_table1 = pd.DataFrame(table_rows)
save_path_xls = f"{OUTPUT_DIR}/Manuscript_Table_S1_Characteristics.xlsx"

try:
    with pd.ExcelWriter(save_path_xls, engine='openpyxl') as writer:
        df_table1.to_excel(writer, index=False, sheet_name='Table_S1')
    print(f"\n[+] Table S1 successfully exported to: {save_path_xls}")
except Exception as e:
    print(f"\n[!] Export failed: {e}")

# Print preview
print("\n--- TABLE S1 PREVIEW ---")
print(df_table1.to_string(index=False))


Generating Translated Manuscript Supplementary Table S1...
[*] Final Cohort Size: N = 75 (Favorable: 35, Unfavorable: 40)

[+] Table S1 successfully exported to: /content/drive/MyDrive/avc/seed/outputs/table_s1/Manuscript_Table_S1_Characteristics.xlsx

--- TABLE S1 PREVIEW ---
                 Characteristic     Overall Favorable Outcome (mRS <= 3) Unfavorable Outcome (mRS > 3)
                 Total Patients   75 (100%)                   35 (46.7%)                    40 (53.3%)
         Age (years), Mean (SD) 68.8 (14.5)                  63.7 (15.0)                   73.2 (12.7)
          Biological Sex, N (%)                                                                       
                         - Male  38 (50.7%)                   18 (51.4%)                    20 (50.0%)
                       - Female  37 (49.3%)                   17 (48.6%)                    20 (50.0%)
Self-declared Race/Color, N (%)                                                                       
